[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bme5014-deeplearning/bme5014-deeplearning/blob/main/F26/recitation_0/Rec-0.3_NumPy_Fundamentals_BME5014.ipynb)


# Recitation 0.3: NumPy Fundamentals for Biomedical Deep Learning (BME5014)

**Prerequisite:** Recitation 0.1 (Python Fundamentals).

Every ECG recording, every X-ray, every MRI volume, and every batch of patient data you feed to a deep learning model is stored as an **array of numbers**. NumPy is the library that makes working with such arrays fast and convenient, and **PyTorch tensors copy NumPy's syntax almost exactly**: what you learn here carries over directly.

In this notebook you will learn:
- Why we need NumPy (speed and clarity)
- Array types (scalar, vector, matrix, tensor) and what their shapes mean for biomedical data
- Creating arrays and choosing data types
- Elementwise operations
- Indexing, slicing, views vs copies, and boolean masking
- Conditional selection with `np.where`
- Aggregations along an axis
- Broadcasting
- Combining and reshaping arrays
- Elementwise, matrix, dot, and outer products

**How to use this notebook:** run every cell, read the output, then change the values and *predict* what will happen before you re-run. Short "Your turn" prompts appear throughout.


## Coming from MATLAB?

If you took a MATLAB course, you already think in arrays. NumPy feels similar, with a few important differences:

| Task | MATLAB | NumPy |
|---|---|---|
| First element | `x(1)` | `x[0]` (indexing starts at **0**) |
| Last element | `x(end)` | `x[-1]` |
| Elements 2 to 4 | `x(2:4)` | `x[1:4]` (the end index is **excluded**) |
| Size | `size(A)` | `A.shape` |
| Zeros matrix | `zeros(2,3)` | `np.zeros((2, 3))` |
| Evenly spaced | `linspace(0,1,5)` | `np.linspace(0, 1, 5)` |
| Elementwise multiply | `A .* B` | `A * B` |
| Matrix multiply | `A * B` | `A @ B` |
| Transpose | `A'` | `A.T` |
| Sum down the columns | `sum(A, 1)` | `np.sum(A, axis=0)` |
| Stack vertically | `[A; B]` | `np.vstack([A, B])` |
| Indices where true | `find(A > 50)` | `np.where(A > 50)` |

Two traps to remember:
1. **`*` is elementwise in NumPy.** Matrix multiplication is `@`.
2. **Slices are views, not copies.** Changing a slice changes the original array (MATLAB copies). We demonstrate this in Section 7.

(The official guide [NumPy for MATLAB users](https://numpy.org/doc/stable/user/numpy-for-matlab-users.html) has a longer table.)


## 1. Why NumPy?

Python lists are flexible, but looping over them element by element is **slow**. NumPy stores numbers in one compact block of memory and runs the loop in fast compiled code. Writing an operation on a whole array at once, without an explicit Python loop, is called **vectorization**.

Let's compute the **RMS amplitude** (root-mean-square, a standard measure of signal strength) of a long ECG-like signal with 1,000,000 samples, first with a plain Python loop, then with NumPy.


In [ ]:
import time
import numpy as np

rng = np.random.default_rng(0)
ecg = rng.standard_normal(1_000_000)    # random numbers standing in for 1,000,000 ECG samples
ecg_list = ecg.tolist()                 # the same data as a plain Python list


def rms_python(x):
    total = 0.0
    for v in x:                         # explicit Python loop
        total += v * v
    return (total / len(x)) ** 0.5


def rms_numpy(x):
    return np.sqrt(np.mean(x ** 2))     # one line, no loop


def best_time(func, x, repeats=5):
    times = []
    for _ in range(repeats):                 # repeat and keep the fastest run (fairer timing)
        t0 = time.perf_counter()
        result = func(x)
        times.append(time.perf_counter() - t0)
    return result, min(times)


r_py, t_py = best_time(rms_python, ecg_list)
r_np, t_np = best_time(rms_numpy, ecg)

print(f"Python loop: RMS = {r_py:.6f}  ({t_py * 1000:.1f} ms)")
print(f"NumPy      : RMS = {r_np:.6f}  ({t_np * 1000:.1f} ms)")
print(f"Same answer: {np.isclose(r_py, r_np)} | NumPy was about {t_py / t_np:.0f}x faster")


Same result, one line instead of five, and much faster (the exact factor depends on the machine; typically 10 to 100 times). With real recordings (hours of multichannel data) or image datasets (thousands of images) this difference decides whether your code finishes in seconds or in hours.


## 2. What is NumPy and how do we import it?

NumPy is a Python library for numerical computing on large multi-dimensional arrays. Its core object is `np.ndarray`. The import is almost always aliased as `np`.


In [ ]:
import numpy as np
print(f"NumPy version: {np.__version__}")

x = np.array([72, 75, 78])
print(type(x))


## 3. Array types and shapes

The number of axes of an array is called its **ndim**, and the size along each axis is its **shape**. In deep learning you will constantly reason about shapes, and **shape mismatches are the most common error you will meet**, so get into the habit of printing `.shape`.

| Name | ndim | Biomedical example | Typical shape |
|---|---|---|---|
| Scalar | 0 | body temperature | `()` |
| Vector | 1 | one ECG lead | `(5000,)` |
| Matrix | 2 | grayscale X-ray, or multichannel EEG | `(256, 256)`, `(19, 1000)` |
| Tensor | 3+ | MRI volume, or a batch of images | `(64, 256, 256)`, `(32, 1, 256, 256)` |


In [ ]:
temperature = np.array(36.6)                  # scalar
ecg_lead = np.zeros(5000)                     # vector: one ECG lead, 5000 samples
eeg = np.zeros((19, 1000))                    # matrix: 19 EEG channels x 1000 time points
xray = np.zeros((256, 256))                   # matrix: grayscale X-ray, height x width
mri = np.zeros((64, 256, 256))                # 3-D tensor: MRI volume, slices x height x width
batch = np.zeros((32, 1, 256, 256))           # 4-D tensor: batch of 32 grayscale images (N, C, H, W)

for name, arr in [("temperature", temperature), ("ecg_lead", ecg_lead), ("eeg", eeg),
                  ("xray", xray), ("mri", mri), ("batch", batch)]:
    print(f"{name:12s} ndim={arr.ndim}  shape={arr.shape}")


## 4. Creating arrays

Below are the most common ways to create an array. We define a small helper that prints an array together with its shape and data type.


In [ ]:
def show(arr, name):
    print(f"{name}:")
    print(arr)
    print("shape:", arr.shape, "| dtype:", arr.dtype)
    print("-" * 40)


# From a Python list (1-D) and a list of lists (2-D)
show(np.array([72, 75, 78, 74]), "1-D array from a list")
show(np.array([[1, 2, 3],
                [4, 5, 6]]), "2-D array from a list of lists")


In [ ]:
# Arrays filled with a constant
show(np.zeros((2, 3)), "zeros (2x3)")
show(np.ones((3, 2)), "ones (3x2)")
show(np.full((2, 4), 7), "all 7s (2x4)")
show(np.eye(3), "identity matrix (3x3)")

# Same shape as another array
template = np.array([[1, 2, 3],
                     [4, 5, 6]])
show(np.zeros_like(template), "zeros_like(template)")
show(np.ones_like(template), "ones_like(template)")


**Ranges and time axes.** `np.arange` is like Python's `range` but returns an array; `np.linspace` gives evenly spaced points. For signals you will often build a **time axis** from the sampling rate `fs`.


In [ ]:
show(np.arange(0, 10, 2), "arange(0, 10, 2): start, stop (excluded), step")
show(np.linspace(0, 1, 5), "linspace(0, 1, 5): 5 points including both ends")

fs = 250                        # sampling rate in Hz
t = np.arange(2 * fs) / fs      # 2 seconds sampled at 250 Hz -> 500 time stamps
print("t.shape:", t.shape, "| first values:", t[:4], "| last value:", t[-1])


In [ ]:
import matplotlib.pyplot as plt

signal = np.sin(2 * np.pi * 1.2 * t)    # a 1.2 Hz wave = 72 oscillations per minute

plt.figure(figsize=(8, 2.5))
plt.plot(t, signal)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title("A 1.2 Hz sine wave (about 72 beats per minute)")
plt.tight_layout()
plt.show()


**Random arrays.** Random numbers are used everywhere in deep learning (weight initialization, noise, data splits). Use a generator with a **seed** so that your results are reproducible. (Older code uses `np.random.rand`; you will see both.)


In [ ]:
rng = np.random.default_rng(seed=42)

show(rng.random((2, 3)), "uniform random numbers in [0, 1) (2x3)")
show(rng.integers(0, 10, size=(3, 3)), "random integers from 0 to 9 (3x3)")
show(rng.normal(loc=0.0, scale=0.1, size=5), "Gaussian noise, mean 0, std 0.1 (5,)")


## 5. Data types

Every array holds items of **one single data type** (`dtype`). Mixing integers and floats gives floats. You choose the dtype at creation (`dtype=`) or convert with `.astype(...)`.

The ones you will meet most often:

| dtype | Where you will see it |
|---|---|
| `float64` | NumPy's default for decimal numbers |
| `float32` | The default for PyTorch tensors and neural networks |
| `int64` | Integer labels, indices |
| `uint8` | 8-bit images (pixel values 0 to 255) |
| `bool` | Masks (`np.bool_` in NumPy) |


In [ ]:
a = np.array([1, 2, 3])
b = np.array([1.0, 2, 3])                      # one float turns the whole array into floats
c = np.array([1, 2, 3], dtype=np.float32)      # choose the dtype at creation
d = a.astype(np.float32)                       # or convert afterwards

for name, arr in [("a", a), ("b", b), ("c", c), ("d", d)]:
    print(name, arr.dtype)


**A classic image-processing bug: `uint8` overflow.** A `uint8` pixel can only hold 0 to 255. If you brighten an 8-bit image carelessly, values *wrap around* instead of stopping at 255:


In [ ]:
img8 = np.array([100, 200, 250], dtype=np.uint8)     # three pixels of an 8-bit image

print("img8 + 10             :", img8 + 10)                              # 250 + 10 wraps around to 4!
print("convert first, add    :", img8.astype(np.int32) + 10)             # safe, but exceeds 255
print("convert, add and clip :", np.clip(img8.astype(np.int32) + 10, 0, 255))  # stops at 255

# Neural networks usually want float32 values between 0 and 1:
print("scaled to [0, 1]      :", img8.astype(np.float32) / 255.0)


## 6. Elementwise operations

Arithmetic between two arrays of the **same shape** is done element by element. Operations with a single number (a scalar) are applied to every element.


In [ ]:
systolic = np.array([118, 135, 150])      # mmHg, three patients
diastolic = np.array([76, 88, 95])

print("pulse pressure (S - D):", systolic - diastolic)
print("mean arterial pressure:", (diastolic + (systolic - diastolic) / 3).round(1))
print("systolic > 130        :", systolic > 130)          # comparisons give boolean arrays


In [ ]:
# Scalar operations apply to every element
temps_f = np.array([97.0, 98.6, 101.2, 103.1])
print("Celsius :", (temps_f - 32) * 5 / 9)

ecg_mv = np.array([0.1, 0.5, 1.2, 0.3])
print("in uV   :", ecg_mv * 1000)

# Mathematical functions work elementwise too
x = np.array([1.0, 4.0, 9.0])
print("sqrt(x):", np.sqrt(x), "| log(x):", np.log(x).round(3), "| x ** 2:", x ** 2)
print("exp([0, 1, 2]):", np.exp(np.array([0.0, 1.0, 2.0])).round(2))


**Dividing by zero** does not crash NumPy; it produces `inf` or `nan` (and usually a warning). A realistic case: standardizing a "dead" channel whose signal is constant, so its standard deviation is 0. In deep learning code you will often see a tiny number such as `1e-8` added to a denominator to avoid this.


In [ ]:
x = np.array([1.0, 2.0, 0.0])
with np.errstate(divide="ignore"):            # silence the warning for this demo
    print("1 / x:", 1 / x)

flat = np.array([5.0, 5.0, 5.0])              # a dead channel: constant signal
std = flat.std()                              # 0.0

with np.errstate(invalid="ignore"):
    print("z-score without epsilon:", (flat - flat.mean()) / std)          # 0 / 0 -> nan
print("z-score with epsilon   :", (flat - flat.mean()) / (std + 1e-8))      # 0 / tiny -> 0


## 7. Indexing, slicing, views and copies

NumPy gives constant-time access to any element. For a 2-D array you give the row first, then the column: `A[row, col]`. Negative indices count from the end.

We will use a tiny 6x6 array as a **grayscale image**: each number is a pixel intensity.


In [ ]:
img = np.arange(0, 180, 5).reshape(6, 6)
show(img, "img (6x6 grayscale image)")

print("pixel at row 1, col 2:", img[1, 2])
print("last pixel           :", img[-1, -1])
print("last column          :", img[:, -1])


**Slicing** extracts a sub-region. The general form is `A[start:stop:step, start:stop:step]`, with one entry per axis, and the `stop` index is **excluded**. A colon on its own (`:`) means "everything along this axis".


In [ ]:
show(img[0:2, :], "first two rows")
show(img[:, 1:3], "columns 1 and 2")
show(img[:, ::2], "every second column (a simple 2x downsampling)")

roi = img[1:4, 2:5]
show(roi, "region of interest: rows 1-3, columns 2-4")


### Views vs copies

**A slice is a view, not a copy.** It shares memory with the original array, so changing the slice changes the original. This is a very common source of bugs. Use `.copy()` when you need an independent array.


In [ ]:
roi = img[1:4, 2:5]          # a VIEW of img
roi[0, 0] = 999
print("img[1, 2] after editing the slice:", img[1, 2])     # the original changed!

img[1, 2] = 40                # restore the original value

roi_copy = img[1:4, 2:5].copy()      # an independent COPY
roi_copy[0, 0] = 999
print("img[1, 2] after editing the copy :", img[1, 2])     # unchanged

print("shares memory with img? view:", np.shares_memory(roi, img), "| copy:", np.shares_memory(roi_copy, img))


You can also assign to a slice, for example to paint a border or blank out a row:


In [ ]:
edited = img.copy()
edited[0, :] = 0         # black top row
edited[:, -1] = 255      # white right column
show(edited, "edited copy")


**Your turn:** extract the bottom-right 3x3 block of `img` and compute its mean (use `.mean()`).


In [ ]:
# your code here


## 8. Boolean masking

A comparison such as `img > 100` produces a **boolean mask** with the same shape as the array. Indexing with the mask selects only the elements where it is `True`. In image analysis this is the basis of **thresholding and segmentation**.


In [ ]:
mask = img > 100
show(mask, "mask: img > 100")

print("bright pixels:", img[mask])
print("how many     :", mask.sum(), "(True counts as 1)")
print("fraction     :", round(mask.mean(), 3))


Combine conditions with `&` (and), `|` (or), `~` (not). Each condition **must be in parentheses**, and you must use these symbols, not the words `and` / `or`.


In [ ]:
mid = (img > 40) & (img < 120)
print("pixels between 40 and 120:", img[mid])

# A binary segmentation mask: 1 = bright region, 0 = background
binary = (img > 100).astype(np.uint8)
show(binary, "binary mask")


You can also pick out exact positions by giving a list of row indices and a list of column indices. They are paired up, so this selects individual pixels, **not** a block:


In [ ]:
rows = [0, 2]
cols = [1, 2]
print("pixels (0,1) and (2,2):", img[rows, cols])


**Your turn:** what fraction of the pixels in `img` is between 20 and 60 (inclusive)?


In [ ]:
# your code here


## 9. Conditional selection with `np.where`

`np.where(condition)` returns the **indices** where the condition is true. `np.where(condition, value_if_true, value_if_false)` builds a **new array** by choosing between two options for every element.


In [ ]:
systolic = np.array([110, 125, 135, 150, 98, 180])

idx = np.where(systolic > 140)
print("indices where systolic > 140:", idx)
print("those values                :", systolic[idx])

print("flag (1 = high)             :", np.where(systolic >= 140, 1, 0))


In [ ]:
# Nested conditions: classify each reading into 0 = normal, 1 = elevated, 2 = hypertension
category = np.where(systolic < 120, 0,
                    np.where(systolic < 140, 1, 2))
print("systolic:", systolic)
print("category:", category)


**Handling sensor artifacts.** Physiological recordings contain impossible values (a loose electrode, a movement). We can replace them with `np.nan` ("not a number") and use the `nan`-aware functions, which simply ignore them.


In [ ]:
hr = np.array([72, 75, 300, 78, 0, 74], dtype=float)    # 300 and 0 bpm are sensor artifacts

clean_hr = np.where((hr > 220) | (hr < 20), np.nan, hr)
print("cleaned          :", clean_hr)
print("mean, raw        :", round(np.mean(hr), 2))
print("mean, ignoring nan:", np.nanmean(clean_hr))


## 10. Aggregations and axes

Functions like `sum`, `mean`, `std`, `min`, `max` collapse an array to a single number, or to a smaller array when you give an **axis**.

Here each **row is a patient** and each **column is a time point** (heart rate in bpm):


In [ ]:
hr = np.array([[72, 75, 78, 74],
               [88, 90, 85, 87],
               [65, 64, 66, 68]])
show(hr, "heart rate: 3 patients x 4 time points")

print("overall mean:", hr.mean(), "| std:", hr.std().round(2), "| min:", hr.min(), "| max:", hr.max())


In [ ]:
print("axis=0, mean over patients (one value per time point):", hr.mean(axis=0).round(2))
print("axis=1, mean over time (one value per patient)        :", hr.mean(axis=1).round(2))


**Rule of thumb: the axis you give is the axis that gets collapsed.**
- `axis=0` collapses the rows, leaving one value per column.
- `axis=1` collapses the columns, leaving one value per row.

`keepdims=True` keeps the collapsed axis with size 1. This is very useful for broadcasting in the next section.


In [ ]:
row_mean = hr.mean(axis=1)
row_mean_kd = hr.mean(axis=1, keepdims=True)
print("without keepdims:", row_mean.shape)
print("with keepdims   :", row_mean_kd.shape)


In [ ]:
# Conditional aggregation
print("mean of readings above 80 bpm:", hr[hr > 80].mean())

# argmax / argmin give POSITIONS instead of values
print("flat index of the maximum    :", np.argmax(hr), "(array is flattened first)")
print("patient with max HR, per time point:", np.argmax(hr, axis=0))
print("time point of max HR, per patient  :", np.argmax(hr, axis=1))


**Your turn:** at which time point (column) is the mean heart rate across patients the highest?


In [ ]:
# your code here


## 11. Broadcasting

**Broadcasting** lets NumPy combine arrays of *different* shapes without copying data. You already used it when adding a scalar to an array. The rule: NumPy compares the shapes starting from the **last** axis, and for each pair of sizes they must be **equal, or one of them must be 1** (a missing axis counts as 1).


In [ ]:
A = np.array([[1, 2, 3],
              [4, 5, 6]])               # shape (2, 3)

row = np.array([10, 20, 30])            # shape (3,)
print("A + 10 :\n", A + 10)
print("A + row (3 matches 3):\n", A + row)


In [ ]:
bad = np.array([100, 200])              # shape (2,)  ->  3 vs 2 does not match

try:
    A + bad
except ValueError as e:
    print("Error:", e)

# Reshape to a column (2, 1) so that 2 matches 2 and 1 stretches to 3
col = bad.reshape(2, 1)                 # same as bad[:, np.newaxis]
print("A + column:\n", A + col)


**Why this matters: normalization.** Before feeding data to a network you often standardize each patient's (or channel's) values to mean 0 and standard deviation 1 (a *z-score*). That is a broadcasting operation with `keepdims=True`:


In [ ]:
mean_per_patient = hr.mean(axis=1, keepdims=True)     # shape (3, 1)
std_per_patient = hr.std(axis=1, keepdims=True)       # shape (3, 1)

z = (hr - mean_per_patient) / std_per_patient         # (3, 4) - (3, 1) -> (3, 4)
show(z.round(2), "z-scored heart rates")
print("mean per patient is now ~0:", np.allclose(z.mean(axis=1), 0))
print("std per patient is now 1  :", np.allclose(z.std(axis=1), 1))


In [ ]:
# Forgetting keepdims is a classic mistake: shapes (3, 4) and (3,) do not broadcast
try:
    hr - hr.mean(axis=1)
except ValueError as e:
    print("Error:", e)


**Your turn:** subtract from `hr` the mean of each *time point* (the mean over patients) so that every column has mean 0. Which axis do you collapse, and do you need `keepdims`?


In [ ]:
# your code here


## 12. Combining and reshaping arrays

### Concatenating and stacking

- `np.concatenate` joins arrays **along an existing axis**.
- `np.stack` joins arrays along a **new axis**.

Shapes must match in every axis except the one you concatenate along.


In [ ]:
lead_I = np.array([0.1, 0.5, 1.2, 0.3])
lead_II = np.array([0.2, 0.6, 1.0, 0.4])

show(np.concatenate([lead_I, lead_II]), "concatenate: one long signal")
show(np.stack([lead_I, lead_II], axis=0), "stack axis=0: two channels (2, 4)")
show(np.stack([lead_I, lead_II], axis=1), "stack axis=1: (4, 2)")


In [ ]:
trial1 = np.array([[1, 2, 3],
                   [4, 5, 6]])          # 2 channels x 3 time points
trial2 = trial1 * 10

show(np.concatenate([trial1, trial2], axis=1), "axis=1: a longer recording (2, 6)")
show(np.concatenate([trial1, trial2], axis=0), "axis=0: more channels (4, 3)")
show(np.stack([trial1, trial2]), "stack: a batch of 2 trials (2, 2, 3)")

# Shortcuts: np.vstack is concatenate along axis 0, np.hstack along axis 1
print(np.vstack([trial1, trial2]).shape, np.hstack([trial1, trial2]).shape)


### Reshaping

`reshape` changes the *shape* without changing the data. The total number of elements must stay the same, and one dimension can be `-1`, meaning "work it out for me".


In [ ]:
X = np.arange(12)
show(X.reshape(3, 4), "reshape(3, 4)")
show(X.reshape(2, 6), "reshape(2, 6)")
show(X.reshape(3, -1), "reshape(3, -1)  ->  (3, 4)")

Y = X.reshape(3, 4)
show(Y.T, "transpose Y.T: rows and columns swapped")
show(Y.flatten(), "flatten: back to 1-D")


In [ ]:
# Typical deep learning reshapes
patch = np.arange(16).reshape(4, 4)
print("flatten an image patch into a feature vector:", patch.flatten().shape)

sig = np.arange(5)
print("add a batch axis:", sig[np.newaxis, :].shape, "| add a channel axis:", sig[:, np.newaxis].shape)


**MATLAB users, careful:** MATLAB fills reshaped arrays *column by column*, NumPy fills them *row by row*. `reshape(0:5, 2, 3)` in MATLAB gives `[0 2 4; 1 3 5]`, while NumPy gives:


In [ ]:
print(np.arange(6).reshape(2, 3))


## 13. All things multiplication

Three different products appear constantly. Mixing them up is a very common mistake.


In [ ]:
A = np.array([[1, 2],
              [3, 4]])
B = np.array([[5, 6],
              [7, 8]])

show(A * B, "elementwise product (A * B)")
show(A @ B, "matrix product (A @ B)")
show(np.matmul(A, B), "matrix product (np.matmul(A, B)): same as @")
show(B @ A, "matrix product (B @ A): order matters!")


**Matrix multiplication** needs the inner sizes to match: `(n, m) @ (m, p)` gives `(n, p)`. This is the core computation of a neural network layer: `output = W @ x + b`. The next cell is what a layer with 3 inputs and 2 outputs does for one patient, and then for a whole batch (this is what `nn.Linear` from Recitation 0.2 computes):


In [ ]:
x = np.array([70.0, 120.0, 5.5])       # 3 clinical features of one patient (weight, systolic BP, glucose)
W = np.array([[0.01, 0.02, -0.1],
              [-0.02, 0.01, 0.3]])      # layer weights: 2 outputs x 3 inputs
b = np.array([0.1, -0.2])               # biases: one per output

y = W @ x + b
print("one patient :", y.round(3), "| shape", y.shape)

X = np.array([[70.0, 120.0, 5.5],
              [82.0, 135.0, 6.1],
              [60.0, 110.0, 4.9],
              [95.0, 150.0, 7.2]])      # a batch of 4 patients: (4, 3)

Y = X @ W.T + b                         # (4, 3) @ (3, 2) -> (4, 2), then b is broadcast over the rows
print("batch of 4  :", Y.shape)


**Outer product.** `np.outer(u, v)` multiplies every element of `u` with every element of `v`, giving a matrix. Example: total dose in mg for every combination of patient weight and dose level (mg per kg):


In [ ]:
weights_kg = np.array([50, 70, 90])
dose_mg_per_kg = np.array([0.5, 1.0])

show(np.outer(weights_kg, dose_mg_per_kg), "total dose (mg): rows = patients, columns = dose levels")


**Dot product.** The dot product of two vectors gives a **single number**:

$$\mathbf{u}\cdot\mathbf{v} = \sum_i u_i v_i = \|\mathbf{u}\|\,\|\mathbf{v}\|\cos\theta$$

where $\theta$ is the angle between the vectors. It is largest when the vectors point in the same direction and **zero when they are orthogonal**, so it measures **similarity**. Dividing by the lengths gives the *cosine similarity* (between -1 and 1), used for example to compare feature vectors of two patients or a signal with a template.


In [ ]:
u = np.array([1, 2, 3])
v = np.array([2, 4, 6])        # same direction as u
w = np.array([3, 0, -1])       # orthogonal to u

print("u . v =", np.dot(u, v), "| u . w =", np.dot(u, w))

def cosine_similarity(a, b):
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))

print("cosine(u, v) =", round(cosine_similarity(u, v), 3))
print("cosine(u, w) =", round(cosine_similarity(u, w), 3))


## 14. Putting it together: preprocessing a noisy multichannel recording

Let's use many of the tools above on a realistic task. We simulate a 3-channel recording (2 seconds at 250 Hz) with a different amplitude and baseline offset on each channel, measurement noise, and one movement artifact. Then we remove the offsets, standardize every channel, and flag the artifact.


In [ ]:
rng = np.random.default_rng(0)
fs = 250
t = np.arange(2 * fs) / fs                          # (500,)

amplitudes = np.array([[1.0], [0.6], [0.8]])        # (3, 1): one amplitude per channel
offsets = np.array([[2.0], [-1.0], [0.5]])          # (3, 1): baseline offset per channel

clean = amplitudes * np.sin(2 * np.pi * 1.2 * t)    # broadcasting: (3, 1) * (500,) -> (3, 500)
noise = rng.normal(0, 0.1, size=clean.shape)        # (3, 500)
recording = clean + offsets + noise
recording[1, 100] += 3.0                            # inject a movement artifact in channel 1

print("recording shape:", recording.shape, "(channels x time)")


In [ ]:
# 1) Remove each channel's baseline and standardize it (broadcasting with keepdims)
mean = recording.mean(axis=1, keepdims=True)         # (3, 1)
std = recording.std(axis=1, keepdims=True)           # (3, 1)
z = (recording - mean) / std                         # (3, 500)

# 2) Flag samples that are extreme in any channel (boolean mask)
artifact_mask = np.abs(z) > 4
channel_idx, sample_idx = np.where(artifact_mask)
print("artifact found in channel", channel_idx, "at sample", sample_idx, "(t =", t[sample_idx], "s)")

# 3) Replace the artifact by nan and compute statistics that ignore it
z_clean = np.where(artifact_mask, np.nan, z)
print("samples removed:", np.isnan(z_clean).sum())
print("channel stds after cleaning:", np.nanstd(z_clean, axis=1).round(2))


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharex=True)

for ch in range(3):
    axes[0].plot(t, recording[ch], label=f"channel {ch}")
axes[0].set_title("Raw recording: different offsets, noise, and one artifact")
axes[0].set_ylim(-2, 4.8)                       # leave headroom for the legend
axes[0].legend(loc="upper right", ncol=3)

for ch in range(3):
    axes[1].plot(t, z_clean[ch], label=f"channel {ch}")
axes[1].plot(t[sample_idx], z[channel_idx, sample_idx], "rx", markersize=10, label="removed artifact")
axes[1].set_title("After standardization and artifact removal")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylim(-2.5, 8)
axes[1].legend(loc="upper right", ncol=4)

plt.tight_layout()
plt.show()


## Common NumPy pitfalls

- **Views vs copies:** slices share memory with the original. Use `.copy()` when you need an independent array.
- **Wrong axis:** the axis you pass is the axis that is collapsed. Check the result's `.shape`.
- **`(n,)` is not `(n, 1)`:** a 1-D array has no "column" orientation. Use `reshape` or `np.newaxis` when broadcasting.
- **`*` vs `@`:** `*` multiplies elementwise, `@` is matrix multiplication.
- **`uint8` overflow:** convert images to a larger or floating-point type before doing arithmetic.
- **`and` / `or` on arrays:** use `&` / `|` with parentheses around each condition.


## Wrap-Up

| Concept | Key idea |
|---|---|
| **Shape / ndim** | Always know the shape of your data; most bugs are shape bugs |
| **Creating arrays** | `array`, `zeros`, `ones`, `arange`, `linspace`, `random` |
| **dtype** | `float32` for networks, `uint8` for images; watch out for overflow |
| **Elementwise operations** | Work on whole arrays at once; no Python loops |
| **Slicing** | `A[start:stop:step]` per axis; slices are **views** |
| **Boolean masks / `np.where`** | Select, count, or replace elements by condition |
| **Axis operations** | `axis` = the dimension that gets collapsed; `keepdims=True` for broadcasting |
| **Broadcasting** | Compare shapes from the last axis; sizes must be equal or 1 |
| **Combine / reshape** | `concatenate`, `stack`, `reshape`, `.T`, `flatten` |
| **Products** | `*` elementwise, `@` matrix, `np.outer`, `np.dot` |

**Next steps:**
- Re-run the notebook, change the numbers, and predict each output before you run the cell.
- Everything here works almost identically on **PyTorch tensors**, which you will use from the next part of the course on.

**Further reading:**
- [NumPy: the absolute basics for beginners](https://numpy.org/doc/stable/user/absolute_beginners.html)
- [NumPy: broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html)
- [NumPy for MATLAB users](https://numpy.org/doc/stable/user/numpy-for-matlab-users.html)
